# Task 3 — Notebook 02: Baseline climatology + 2019 anomalies

**Baseline:** ISO calendar weeks, years **2015–2021**, μ and σ per (`iy`, `ix`, `iso_week`).

**Event:** 2019 weeks whose metadata dates fall in `event_window` (default April–July 2019). **z-score** = (obs − μ) / (σ + floor).

**Outputs:** `data/processed/task3/smap_climatology.parquet`, `data/processed/task3/smap_anomaly_2019.parquet`

**Runtime:** repeated Parquet reads — run after notebook 01.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Repo root not found")
sys.path.insert(0, str(REPO_ROOT))

from src.io.smap_weekly_parquet import event_week_columns_2019, load_smap_year_metadata
from src.modeling.task3_smap_anomalies import (
    baseline_climatology_iso_weeks,
    compute_event_anomalies,
)

with open(REPO_ROOT / "configs" / "task3_soil_moisture.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

b0, b1 = int(cfg["baseline"]["period"][0]), int(cfg["baseline"]["period"][1])
ev = cfg["event_window"]
start, end = ev["start_date"], ev["end_date"]

panel_pq = REPO_ROOT / cfg["output"]["processed_dir"] / "task3_pixel_panel.parquet"
if not panel_pq.is_file():
    raise FileNotFoundError(f"Run notebook 01 first: missing {panel_pq}")
pixels = pd.read_parquet(panel_pq)

meta19 = load_smap_year_metadata(REPO_ROOT, 2019)
event_specs = event_week_columns_2019(meta19, start, end)
iso_weeks = sorted({w for (_, _, w) in event_specs})
print("Event ISO weeks:", iso_weeks[0], "…", iso_weeks[-1], "count", len(iso_weeks))

clim = baseline_climatology_iso_weeks(
    REPO_ROOT, range(b0, b1 + 1), iso_weeks, pixels[["iy", "ix"]]
)
out_dir = REPO_ROOT / cfg["output"]["processed_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
clim_pq = out_dir / "smap_climatology.parquet"
clim.to_parquet(clim_pq, index=False)
print("Wrote", clim_pq.relative_to(REPO_ROOT), "rows", len(clim))

anom = compute_event_anomalies(REPO_ROOT, 2019, event_specs, clim, pixels)
anom_pq = out_dir / "smap_anomaly_2019.parquet"
anom.to_parquet(anom_pq, index=False)
print("Wrote", anom_pq.relative_to(REPO_ROOT), "rows", len(anom))
